# Detector ternário de mosquitos: treino, teste e aplicação

Este notebook organiza o fluxo principal do detector ternário do repositório. Ele usa os scripts-fonte existentes para manter o protocolo, os hashes e os guardas de integridade num só lugar, sem copiar uma segunda implementação do modelo.

As classes são `no_mosquito`, `other_mosquito` e `aedes_aegypti`. O modelo é experimental: a avaliação OOF mede desempenho nos grupos deste corpus, não comprova desempenho em campo nem independência biológica.

**Fluxo:** conferir os dados locais → consultar a avaliação já registrada → opcionalmente treinar e avaliar uma rodada nova → aplicar o modelo final a um WAV real.

> **Custo:** uma rodada nova executa validação cruzada aninhada, auditoria de PCM de todos os arquivos retidos e avaliação OOF. Pode levar bastante tempo. O áudio privado de enxame do Drive é excluído do fluxo atual.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

WORKING_DIR = Path.cwd().resolve()
ROOT_CANDIDATES = [WORKING_DIR, *WORKING_DIR.parents]
ROOT = next(
    (candidate for candidate in ROOT_CANDIDATES
     if (candidate / "train_ternary_nested_models.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Não encontrei a raiz mosquito-wingbeat nos diretórios acima do notebook.")
sys.path.insert(0, str(ROOT))

REFERENCE_RUN = ROOT / "results/ternary_verified_2026_10_14_corrections"
TRAINING_SEGMENTS = ROOT / "results/sex_balanced_reliability_2026_10_01/segments.csv"
TRAINING_FEATURES = ROOT / "data/cache/reliability_2026_10_01/features.npy"
PCM_COLLISION_MAP = ROOT / "results/species_nested_2026_10_13/recording_predictions.csv"

# Mude para True para treinar e avaliar uma NOVA rodada em diretório próprio.
# Rodadas concluídas ou iniciadas nunca são sobrescritas.
RUN_NEW_TRAINING = False
THREADS = 4

# Informe um WAV real para a etapa final. Ex.: Path("data/raw/exemplo.wav")
AUDIO_PATH = None

## 1. Entradas e identidade da execução

O treino não baixa dados automaticamente. Ele requer a tabela de segmentos, as features em cache, o mapa de colisões PCM e os arquivos de áudio locais para a auditoria de bytes/PCM. A rodada nova salva todos os seus artefatos numa pasta nova em `results/`.

In [ ]:
required_inputs = {
    "segmentos anotados": TRAINING_SEGMENTS,
    "features em cache": TRAINING_FEATURES,
    "mapa de colisões PCM": PCM_COLLISION_MAP,
}
missing = [f"{name}: {path}" for name, path in required_inputs.items() if not path.is_file()]
if missing:
    raise FileNotFoundError("Entradas necessárias ausentes:\n- " + "\n- ".join(missing))

print("Raiz:", ROOT)
for name, path in required_inputs.items():
    print(f"{name}: {path.relative_to(ROOT)} ({path.stat().st_size:,} bytes)")
print("Execução nova habilitada:", RUN_NEW_TRAINING)

## 2. Resultado de referência já registrado

A avaliação de referência contém 42.029 janelas e 1.494 grupos em cinco dobras externas. A decisão principal é feita por grupo; janelas sobrepostas não são tratadas como observações independentes. A tabela abaixo lê os artefatos já salvos, sem retreinar nem recalcular métricas.

In [ ]:
reference_metrics_path = REFERENCE_RUN / "event_level_metrics.json"
reference_stats_path = REFERENCE_RUN / "dataset_statistics.json"
if not reference_metrics_path.is_file() or not reference_stats_path.is_file():
    raise FileNotFoundError("A execução de referência ou seus resumos não estão disponíveis.")

reference_metrics = json.loads(reference_metrics_path.read_text())
reference_stats = json.loads(reference_stats_path.read_text())
reference_model = "activity_gated_log_odds_cascade_event"
reference_result = reference_metrics["models"][reference_model]

pd.DataFrame([
    {"Item": "Janelas retidas", "Valor": f"{reference_stats['retained_total']:,}"},
    {"Item": "Grupos retidos", "Valor": f"{sum(reference_stats['group_counts'].values()):,}"},
    {"Item": "Grupos Aedes", "Valor": f"{reference_stats['group_counts']['aedes_aegypti']:,}"},
    {"Item": "Grupos outros mosquitos", "Valor": f"{reference_stats['group_counts']['other_mosquito']:,}"},
    {"Item": "Grupos ruído", "Valor": f"{reference_stats['group_counts']['no_mosquito']:,}"},
    {"Item": "Acurácia", "Valor": f"{reference_result['accuracy']:.2%}"},
    {"Item": "Acurácia balanceada", "Valor": f"{reference_result['balanced_accuracy']:.2%}"},
    {"Item": "Sensibilidade Aedes", "Valor": f"{reference_result['aedes_recall']:.2%}"},
    {"Item": "Precisão Aedes", "Valor": f"{reference_result['aedes_precision']:.2%}"},
    {"Item": "Ruído classificado como Aedes", "Valor": f"{reference_result['integer_counts']['noise_fp_to_aedes']}/{reference_result['integer_counts']['noise_total']}"},
    {"Item": "Outros mosquitos classificados como Aedes", "Valor": f"{reference_result['integer_counts']['other_fp_to_aedes']}/{reference_result['integer_counts']['other_total']}"},
    {"Item": "Ruído classificado como algum mosquito", "Valor": f"{reference_result['integer_counts']['noise_fp_to_any_mosquito']}/{reference_result['integer_counts']['noise_total']}"},
])

A referência é uma avaliação OOF retrospectiva e continua sendo evidência do corpus, não validação independente em novos locais ou aparelhos. As metas operacionais registradas não foram todas atingidas: a cascata teve 23/1.008 erros de outros mosquitos para Aedes, acima do limite amostral de 19.

## 3. Treino e teste OOF de uma rodada nova

Com `RUN_NEW_TRAINING = True`, as funções abaixo executam o treinamento aninhado e, em seguida, a avaliação externa OOF. A rodada registra hashes de código/entradas, particionamento por bloco PCM, seleção interna, modelos congelados e métricas. O teste externo não participa da seleção dos modelos ou limiares.

In [ ]:
from train_ternary_nested_models import train, evaluate

TRAINED_RUN = None
if RUN_NEW_TRAINING:
    run_id = datetime.now(timezone.utc).strftime("ternary_notebook_%Y%m%dT%H%M%SZ")
    TRAINED_RUN = ROOT / "results" / run_id
    train(TRAINED_RUN, threads=THREADS)
    evaluation = evaluate(TRAINED_RUN, threads=THREADS)
    print("Treino e avaliação concluídos:", TRAINED_RUN)
else:
    print(
        "Treino novo não executado. Para criar uma execução reproduzível, "
        "defina RUN_NEW_TRAINING = True e rode esta célula."
    )

### Leitura das métricas e matriz de confusão

Após o treino e o teste da célula anterior, esta célula mostra a avaliação da rodada nova. Se o treino estiver desativado, mostra a matriz OOF da execução de referência.

In [ ]:
ACTIVE_RUN = TRAINED_RUN or REFERENCE_RUN
metrics_file = ACTIVE_RUN / "event_level_metrics.json"
predictions_file = ACTIVE_RUN / "event_predictions.csv"
if not metrics_file.is_file() or not predictions_file.is_file():
    raise FileNotFoundError("A avaliação OOF não está completa para a execução ativa.")

metrics = json.loads(metrics_file.read_text())
results = metrics["models"][reference_model]
counts = results["integer_counts"]
print(f"Execução: {metrics['run_id']}")
print(f"Acurácia: {results['accuracy']:.2%} | balanceada: {results['balanced_accuracy']:.2%}")
print(f"Aedes: {counts['aedes_tp']}/{counts['aedes_total']} detectados")
print(f"Ruído → Aedes: {counts['noise_fp_to_aedes']}/{counts['noise_total']}")
print(f"Outros → Aedes: {counts['other_fp_to_aedes']}/{counts['other_total']}")
print(f"Ruído → algum mosquito: {counts['noise_fp_to_any_mosquito']}/{counts['noise_total']}")

predictions = pd.read_csv(predictions_file)
class_names = ["Sem mosquito", "Outro mosquito", "Aedes aegypti"]
cm = confusion_matrix(
    predictions["true_label"],
    predictions[reference_model + "_pred"],
    labels=[0, 1, 2],
)
display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
display.plot(cmap="Blues", values_format="d", colorbar=False)
plt.title("OOF por grupo — linhas: classe real; colunas: previsão")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 4. Aplicação em uma gravação WAV

A inferência usa o modelo final treinado com todos os dados de desenvolvimento daquela rodada. O código verifica os hashes antes de carregar o modelo, não repete nem completa áudio curto e grava o resumo e as previsões por janela numa pasta nova.

Para aplicar o modelo gerado neste notebook, mantenha `RUN_NEW_TRAINING = True`, informe um WAV em `AUDIO_PATH` na primeira célula e execute as células em ordem. A aplicação do modelo de referência só é liberada se seus hashes continuarem válidos; qualquer divergência exige uma nova rodada congelada.

In [ ]:
if AUDIO_PATH is None:
    print("Defina AUDIO_PATH como o caminho de um WAV real e execute esta célula novamente.")
elif TRAINED_RUN is None:
    print(
        "Nenhum treino novo foi executado nesta sessão. Para preservar a checagem de "
        "integridade do modelo, habilite RUN_NEW_TRAINING, execute as etapas anteriores "
        "e então aplique o áudio."
    )
else:
    from predict_ternary import predict_audio

    audio_path = Path(AUDIO_PATH)
    if not audio_path.is_absolute():
        audio_path = ROOT / audio_path
    if not audio_path.is_file():
        raise FileNotFoundError(f"Gravação não encontrada: {audio_path}")

    application_dir = ACTIVE_RUN / "application_example"
    if application_dir.exists():
        application_dir = ACTIVE_RUN / (
            "application_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        )
    application = predict_audio(TRAINED_RUN, audio_path, application_dir)
    print(json.dumps(application, ensure_ascii=False, indent=2))

## 5. Como interpretar a saída

- `no_mosquito`: o áudio não foi classificado como mosquito.
- `other_mosquito`: candidato a mosquito fora da classe Aedes aegypti.
- `aedes_aegypti`: candidato a Aedes aegypti, sujeito aos erros observados no teste.
- `invalid_audio`: todas as janelas foram rejeitadas por controles de qualidade.

A saída é uma classificação acústica experimental. Não confirma espécie por método biológico, infecção, presença de vetor em campo ou risco epidemiológico. O teste OOF e a aplicação de uma gravação local também não equivalem a uma campanha independente de validação.

### Fontes de implementação e artefatos

- Treinamento/avaliação: `train_ternary_nested_models.py`
- Protocolo, carregamento de dados e integridade: `ternary_protocol.py`
- Métricas por grupo: `ternary_reporting.py`
- Inferência em WAV: `predict_ternary.py`
- Relatório da avaliação: `reports/correcoes_ternarias_2026_10_14.md`
- Exclusão do áudio privado: `reports/exclusao_audio_drive_2026_10_15.md`